# 2D segmentation using CellPose

This notebook demonstrates a complete 2D deep learning segmentation workflow using CellPose. It loads a 2D surface with BioIO and Dask for memory efficiency, visualizes it in Napari's interactive viewer, applies GPU-accelerated Cellpose for automated 2D cell segmentation, and overlays the resulting masks on the original surface for evaluation. It requires bioio, napari, and cellpose with GPU support.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Load and visualize 2D microscopy data** using BioIO and Napari
    - Understand how to efficiently handle large 2D images with Dask arrays
    - Navigate and render 2D surfaces in Napari's viewer

2. **Apply deep learning segmentation to 2D data** using Cellpose
    - Configure Cellpose for 2D segmentation tasks
    - Understand key parameters: `diameter`, `z_axis`, and `do_2D`
    - Leverage GPU acceleration for faster processing

3. **Evaluate and refine segmentation results**
    - Visualize segmentation masks overlaid on original data
    - Identify when parameters need adjustment (e.g., cell diameter)
    - Recognize scenarios requiring model retraining

4. **Develop practical skills for 2D image analysis workflows**
    - Build reproducible analysis pipelines in Jupyter notebooks
    - Work with real biological imaging data (Lund dataset)
    - Understand the relationship between image properties and segmentation parameters

## Load the data

Using [BioIO](https://bioio-devs.github.io/bioio/OVERVIEW.html) with [Dask](https://docs.dask.org/en/stable/) lets us stream the 2D surface lazily so it loads without blowing up memory before sending it to Napari and Cellpose.


To use different image file, comment and uncomment lines by removing and adding # symbol

In [ ]:
from bioio import BioImage

## Two channels image
image_handle = BioImage("../data/Tissue_16x_avg.nd2")

## Channel 1 - Tissue
#image_data = image_handle.dask_data.squeeze()[0]

## Channel 2 - Cells
#image_data = image_handle.dask_data.squeeze()[1]


#-------------------------------
## One channel image
#image_handle = BioImage("../data/Nuclei_avg16x.nd2")
#image_data = image_handle.dask_data.squeeze()


#image_data = image_data.rechunk('auto')

In [24]:
image_data

dask.array<getitem, shape=(2720, 2720), dtype=uint16, chunksize=(2720, 2720), chunktype=numpy.ndarray>

## Visualize in Napari

We'll use [Napari](https://napari.org/stable/) to interactively visualize the 2D volume.

In [16]:
import napari

# Use Dask for lazy loading of the heavy 3D image
# BioImage's .dask_data provides a Dask array without loading into memory
image_data = image_handle.dask_data.squeeze()
image_data

dask.array<getitem, shape=(2720, 2720), dtype=uint16, chunksize=(2720, 2720), chunktype=numpy.ndarray>

In [17]:
# Create a Napari viewer and add the 3D image
viewer = napari.Viewer()
viewer.add_image(image_data, name='Image surface')
napari.run()

## Load and run CellPose in GPU

Cellpose is an open-source deep learning model for general cell and nucleus segmentation in microscopy images. In this notebook, it will be used to segment nuclei in the 2D surface. For details, see the [Cellpose documentation](https://cellpose.readthedocs.io/).

In [18]:
from cellpose import io, models, core

# Initialize Cellpose model with GPU enabled
model = models.CellposeModel(gpu=True)

io.logger_setup() # run this to get printing of progress

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

2026-08-05 10:08:57,670 [INFO] ** TORCH CUDA version installed and working. **
2026-08-05 10:08:57,671 [INFO] >>>> using GPU (CUDA)
2026-08-05 10:08:59,586 [INFO] >>>> loading model C:\Users\ignacio\.cellpose\models\cpsam
creating new log file
2026-08-05 10:09:00,129 [INFO] WRITING LOG OUTPUT TO C:\Users\ignacio\.cellpose\run.log
2026-08-05 10:09:00,130 [INFO] 
cellpose version: 	4.1.1 
platform:       	win32 
python version: 	3.11.15 
torch version:  	2.13.0
2026-08-05 10:09:00,131 [INFO] ** TORCH CUDA version installed and working. **


In [ ]:
# Run Cellpose segmentation in 2D mode
masks, flows, styles = model.eval(
    image_data.compute(),
    diameter=100,  # Adjust based on typical cell size in pixels
    z_axis=None,  # Specify the z-axis for 3D data
    do_3D=False  # Enable 3D segmentation 
)

In [20]:
import napari
viewer = napari.Viewer()
viewer.add_image(image_data, name="Image surface", rendering="attenuated_mip")
viewer.add_labels(masks, name="Cellpose masks", opacity=0.5)
viewer.dims.ndisplay = 2
napari.run()

## Questions

1. Is there any parameter that should be corrected when running CellPose?

2. Do we need to retrain the network?

### Harder questions

1. Can you run Cellpose on the dataset called `lund1051_resampled.tif`? What happens when you choose the right size of the cell nucleus?